# 개별종목 조합B — RandomForest

`기본모델/02.RandomForest.ipynb`과 같은 `models.random_forest.build_random_forest_baseline`을 가져오고
조합B 피처를 주입합니다. 기본모델 코드는 `models/`에 한 번만 존재합니다.
후보·라벨·날짜 그룹 12폴드 실행은 모든 조합이 같은 공통 함수를 사용합니다.


In [1]:
# 1. 기본모델을 가져옵니다.
import sys
from pathlib import Path

import pandas as pd
from IPython.display import display

project_root = Path.cwd().resolve()
while project_root != project_root.parent and not (project_root / "pyproject.toml").is_file():
    project_root = project_root.parent
if not (project_root / "pyproject.toml").is_file():
    raise RuntimeError("프로젝트 루트를 찾지 못했습니다.")
if str(project_root) not in sys.path:
    sys.path.insert(0, str(project_root))

from models.random_forest import build_random_forest_baseline  # noqa: E402

MODEL_NAME = 'RandomForest'
MODEL_BUILDER = build_random_forest_baseline


In [2]:
# 2. 조합B의 피처 값만 지정합니다.
import json

COMBINATION = 'B'
FEATURE_COLUMNS = (
    'ret_5',
    'ret_20',
    'sma_gap_5_20',
    'sma_gap_20_60',
    'rsi_14',
    'dist_high_20',
    'dist_high_60',
)

report_path = project_root / "reports" / "stock_feature_combinations.json"
report = json.loads(report_path.read_text(encoding="utf-8"))
print(f"조합{COMBINATION} 피처:", FEATURE_COLUMNS)
combination_report = report["combinations"].get(COMBINATION)
if combination_report is None:
    print("아직 실측 결과가 없습니다. 아래 공통 실행 명령으로 조합을 평가하세요.")
else:
    panel = combination_report["panel"]
    print("학습 기간:", panel["first_date"], "~", panel["last_date"])
    print("학습 행·종목:", panel["model_rows"], panel["stocks"])
    folds = pd.DataFrame(combination_report["outer_fold_results"])
    model_folds = folds.loc[folds["model"].eq(MODEL_NAME)].reset_index(drop=True)
    fold_columns = [
        "fold", "selected_class_weight", "train_dates", "valid_start", "valid_end",
        "accuracy", "training_majority_baseline_accuracy",
        "accuracy_minus_training_majority_baseline", "macro_f1", "balanced_accuracy",
        "mcc", "pr_auc_macro_ovr", "down_recall", "core_harmonic_mean",
    ]
    display(model_folds.loc[:, fold_columns].round(4))
    metric_columns = [
        "accuracy", "training_majority_baseline_accuracy",
        "accuracy_minus_training_majority_baseline", "macro_f1", "balanced_accuracy",
        "mcc", "pr_auc_macro_ovr", "down_recall", "core_harmonic_mean",
    ]
    display(model_folds.loc[:, metric_columns].mean().to_frame("OOS 폴드 평균").round(4))

# 조합별 노트북이 중복 학습하지 않도록 실제 fit은 공통 실행기에서 한 번 수행합니다.
print("재실행 명령: python scripts/run_stock_model_experiment.py")


조합B 피처: ('ret_5', 'ret_20', 'sma_gap_5_20', 'sma_gap_20_60', 'rsi_14', 'dist_high_20', 'dist_high_60')
학습 기간: 20110127 ~ 20240822
학습 행·종목: 159900 157


,fold,selected_class_weight,train_dates,valid_start,valid_end,accuracy,training_majority_baseline_accuracy,accuracy_minus_training_majority_baseline,macro_f1,balanced_accuracy,mcc,pr_auc_macro_ovr,down_recall,core_harmonic_mean
0,1,balanced,750,20140217,20140514,0.4268,0.5012,-0.0744,0.3640,0.3645,0.0619,0.3632,0.2372,0.3223
1,2,balanced,980,20150123,20150421,0.3537,0.3978,-0.0442,0.3420,0.3449,0.0230,0.3533,0.3150,0.3361
2,3,balanced,1210,20151228,20160328,0.3505,0.3762,-0.0257,0.3488,0.3488,0.0252,0.3554,0.3411,0.3467
3,4,balanced,1439,20161202,20170228,0.4210,0.4617,-0.0407,0.3922,0.3917,0.0901,0.3818,0.3155,0.3706
4,5,balanced,1669,20171113,20180207,0.3768,0.3901,-0.0132,0.3663,0.3670,0.0524,0.3582,0.3268,0.3553
5,6,balanced,1899,20181024,20190118,0.3984,0.3725,0.0259,0.3976,0.3982,0.1000,0.3940,0.3963,0.3974
6,7,balanced,2129,20190930,20191224,0.4180,0.4781,-0.0601,0.3791,0.3813,0.0764,0.3786,0.3295,0.3720
7,8,NaN,2359,20200902,20201130,0.3707,0.3476,0.0231,0.3667,0.3727,0.0580,0.3713,0.3651,0.3675
8,9,balanced,2589,20210806,20211105,0.3684,0.3914,-0.0230,0.3580,0.3638,0.0397,0.3564,0.3061,0.3419
9,10,balanced,2818,20220714,20221012,0.3544,0.3454,0.0090,0.3521,0.3538,0.0301,0.3491,0.3291,0.3448


,OOS 폴드 평균
accuracy,0.3818
training_majority_baseline_accuracy,0.3969
accuracy_minus_training_majority_baseline,-0.0150
macro_f1,0.3670
balanced_accuracy,0.3686
mcc,0.0555
pr_auc_macro_ovr,0.3666
down_recall,0.3362
core_harmonic_mean,0.3587


재실행 명령: python scripts/run_stock_model_experiment.py
